# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Croissant dataset with the [`mlcroissant`](https://mlcommons.github.io/croissant/api/dataset/) library.

### Dataset Source
---
The dataset is described via its Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. We also print summary information.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("Dataset name: ", metadata.name)
print("Description: ", metadata.description)
print("Authors: ", getattr(metadata, 'author', ''))
print("Published: ", getattr(metadata, 'datePublished', ''))


## 2. Data Overview

Review available record sets, fields, and their IDs. All references use the `@id` attribute for reproducibility.


In [ ]:
# Print the available record sets (@id and name)
record_sets = dataset.record_sets
print("Available Record Sets:")
for rset in record_sets:
    print(f"- @id: {rset['@id']} | name: {rset.get('name', '')}")

# Show fields for each record set
for rset in record_sets:
    print(f"\nRecordSet @id: {rset['@id']} | name: {rset.get('name', '')}")
    print("Fields (@id | name | dataType):")
    for field in rset['field']:
        if isinstance(field, dict):
            f = field
        else:
            f = dataset._get_entity(field)  # mlcroissant private for demo
        print(f"  - @id: {f['@id']} | name: {f.get('name','')} | dataType: {f.get('dataType','')}")


## 3. Data Extraction

We'll load data from each record set into a Pandas DataFrame. Refer to the above cells for all valid record set and field `@id`s.


In [ ]:
# List record set @id(s)
record_set_ids = [rset['@id'] for rset in dataset.record_sets]
print("Record set ids:\n", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    # We'll create DataFrames only for non-empty record sets
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records, columns:\n{df.columns.tolist()}")
        display(df.head())

# Select a record set for analysis
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Selected record set for EDA: {selected_record_set_id}")
else:
    selected_record_set_id = None


## 4. Exploratory Data Analysis (EDA)

Below we demonstrate numeric field filtering, normalization, and grouping using only field/column `@id` names for reproducibility.


In [ ]:
import numpy as np

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]

    # Try to choose a numeric field for quant analysis
    # We'll auto-detect the first numeric-looking column
    numeric_field_id = None
    for col in df.columns:
        # Quick dtype float64/int64 or convertible
        try:
            vals = pd.to_numeric(df[col], errors='coerce').dropna()
            if len(vals) > 0:
                numeric_field_id = col
                break
        except Exception:
            continue

    if numeric_field_id is not None:
        print(f"Numeric field selected (by @id): {numeric_field_id}")

        # Filter: take values above a quantile threshold or specific value
        threshold = np.nanmedian(pd.to_numeric(df[numeric_field_id], errors='coerce'))
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (pick first categorical/other field)
        group_field = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            if df[col].dtype == object and df[col].nunique() < df.shape[0] // 2:
                group_field = col
                break
        if group_field is not None:
            grouped_df = (filtered_df.groupby(group_field)[numeric_field_id]
                                     .mean()
                                     .reset_index()
                                     .sort_values(numeric_field_id, ascending=False))
            print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric field detected for EDA.")
else:
    print("No records to analyze.")


## 5. Visualization

Let's visualize the distribution of the selected numeric field (if available) and relationships with the grouping field using `matplotlib` and `seaborn`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    data = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
    sns.histplot(data, bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion

In this notebook, we've demonstrated loading, exploring, and visualizing a mass spectrometry dataset of extracellular vesicle proteins from stimulated human mast cells using the [mlcroissant](https://mlcommons.github.io/croissant/) library, with all data access referencing Croissant `@id` fields. You can extend this EDA further by inspecting additional record sets, visualizing other fields, or performing downstream analysis ready for your bioinformatics or proteomics workflows.
